# Thu nghiem: Overlap-tile blending co cai thien PSNR so voi tile roi rac khong?

File rieng, tach khoi `test_onnx_quality_local.ipynb`, chi tap trung tra loi 1 cau hoi:

> Pipeline ONNX hien tai chia anh thanh cac tile `256x256` **roi rac, khong chong lap**
> (moi tile xu ly doc lap, khong co ngu canh tu tile lan can). Ket qua do duoc tren
> full test set GoPro: **33.25 dB**, thap hon paper (dung `NAFNetLocal` tren nguyen anh,
> co TLC) **33.71 dB** — chenh **0.46 dB**.
>
> Neu dung **tile chong lap (overlap) + blend vung bien** khi ghep lai, ket qua co
> tien gan hon ve phia 33.71 dB khong? Danh doi la gi (thoi gian chay tang len bao nhieu)?

Ket luan se giup quyet dinh: co dang de trien khai overlap-tile cho ban mobile that hay khong.

## Buoc 1. Import + load ONNX session

In [ ]:
import os
import sys
import time
import glob

REPO_ROOT = os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Cac thu vien CUDA/cuDNN da di kem san trong .venv (qua goi torch cu121), nhung
# khong nam trong LD_LIBRARY_PATH mac dinh -> onnxruntime khong tim thay, tu roi
# xuong CPU (rat cham). Them cac thu muc lib cua nvidia-* vao truoc khi tao session.
nvidia_lib_dirs = glob.glob(os.path.join(REPO_ROOT, '.venv/lib/python*/site-packages/nvidia/*/lib'))
if nvidia_lib_dirs:
    os.environ['LD_LIBRARY_PATH'] = ':'.join(nvidia_lib_dirs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

import numpy as np
import onnxruntime as ort
from PIL import Image
import matplotlib.pyplot as plt

ONNX_SIMPLIFIED_PATH = '../Result/onnx/nafnet_gopro_width64_256x256_sim.onnx'  # <-- khop voi convert_to_onnx_local.ipynb
assert os.path.exists(ONNX_SIMPLIFIED_PATH), f'Khong tim thay: {ONNX_SIMPLIFIED_PATH}'

# Uu tien GPU (CUDA) neu co — model width64 kha nang, CPU chay tung tile rat cham
# (~0.8s/tile CPU vs ~0.05s/tile GPU, nhanh hon ~15 lan). Danh gia overlap can rat
# nhieu luot forward nen GPU giup giam thoi gian tu hang gio xuong vai phut.
available_providers = ort.get_available_providers()
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else ['CPUExecutionProvider']
sess = ort.InferenceSession(ONNX_SIMPLIFIED_PATH, providers=providers)
print('Providers dang dung:', sess.get_providers())

TILE = sess.get_inputs()[0].shape[2]  # 256, co dinh boi luc export ONNX
print('Da load ONNX session, tile co dinh:', TILE)

## Buoc 2. Ham dung chung: tile roi rac (baseline) vs tile chong lap + blend

`run_tiled_baseline`: giong het `run_tiled_inference` trong `test_onnx_quality_local.ipynb`
— tile `TILE x TILE` khong chong lap, ghep truc tiep (khong blend).

`run_tiled_overlap`: chia tile voi **stride < TILE** (tao ra vung chong lap giua cac tile
lien ke), moi tile duoc nhan trong so bang mot **window 2D dang tam giac** (thap o bien,
cao o giua) truoc khi cong don vao anh ket qua; vung chong lap se duoc **blend theo trong so**
thay vi bi tile sau de len tile truoc. Vung nao chi co 1 tile phu (khong chong lap — vi du
o goc anh) thi phep chia trong so tu trieu tieu dung boi chuan hoa `out_acc / weight_acc`,
nen khong lam sai lech ket qua o nhung vung do.

In [ ]:
def compute_positions(dim, tile, stride):
    """Danh sach diem bat dau tile doc theo 1 truc, luon co 1 tile neo dung mep cuoi."""
    if dim <= tile:
        return [0]
    positions = list(range(0, dim - tile + 1, stride))
    if positions[-1] != dim - tile:
        positions.append(dim - tile)
    return positions


def make_blend_weight(tile, overlap):
    """Window 2D: bang 1 o giua, giam dan tuyen tinh ve phia bien trong vung overlap."""
    ramp = np.ones(tile, dtype=np.float32)
    if overlap > 0:
        vals = np.linspace(1.0 / (overlap + 1), 1.0, overlap, dtype=np.float32)
        ramp[:overlap] = vals
        ramp[-overlap:] = vals[::-1]
    return np.outer(ramp, ramp)[:, :, None]  # (tile, tile, 1)


def run_tiled_baseline(image_path, sess, tile):
    """Tile roi rac, khong chong lap, khong blend — dung cach pipeline hien tai dang lam."""
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    img_np = np.array(img).astype(np.float32) / 255.0

    pad_h, pad_w = (-h) % tile, (-w) % tile
    img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    H_pad, W_pad = img_padded.shape[:2]
    out_padded = np.zeros_like(img_padded)

    n_tiles = 0
    for y in range(0, H_pad, tile):
        for x in range(0, W_pad, tile):
            tile_in = img_padded[y:y+tile, x:x+tile, :].transpose(2, 0, 1)[None].astype(np.float32)
            tile_out = sess.run(None, {'input': tile_in})[0]
            out_padded[y:y+tile, x:x+tile, :] = tile_out[0].transpose(1, 2, 0)
            n_tiles += 1

    out = np.clip(out_padded[:h, :w, :], 0, 1)
    return img_np, out, n_tiles


def run_tiled_overlap(image_path, sess, tile, overlap):
    """Tile chong lap voi stride = tile - overlap, blend vung chong lap bang trong so."""
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    img_np = np.array(img).astype(np.float32) / 255.0

    pad_h, pad_w = max(0, tile - h), max(0, tile - w)
    img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    H_pad, W_pad = img_padded.shape[:2]

    stride = tile - overlap
    ys = compute_positions(H_pad, tile, stride)
    xs = compute_positions(W_pad, tile, stride)
    weight2d = make_blend_weight(tile, overlap)

    out_acc = np.zeros((H_pad, W_pad, 3), dtype=np.float32)
    weight_acc = np.zeros((H_pad, W_pad, 1), dtype=np.float32)

    for y in ys:
        for x in xs:
            tile_in = img_padded[y:y+tile, x:x+tile, :].transpose(2, 0, 1)[None].astype(np.float32)
            tile_out = sess.run(None, {'input': tile_in})[0][0].transpose(1, 2, 0)
            out_acc[y:y+tile, x:x+tile, :] += tile_out * weight2d
            weight_acc[y:y+tile, x:x+tile, :] += weight2d

    out = out_acc / np.clip(weight_acc, 1e-6, None)
    out = np.clip(out[:h, :w, :], 0, 1)
    return img_np, out, len(ys) * len(xs)


print('Da dinh nghia baseline (khong overlap) va overlap+blend.')

## Buoc 3. So sanh nhanh tren anh demo (`demo/blurry.jpg`)

Chi 1 anh nen chay nhanh — dung de xem qua khac biet truc quan va so tile/thoi gian truoc
khi chay danh gia PSNR quy mo lon o Buoc 5.

In [ ]:
INPUT_IMAGE_PATH = './demo/blurry.jpg'
OVERLAP = 64  # <-- thu doi 32/64/96 de xem anh huong

t0 = time.time()
img_in, out_baseline, n_tiles_baseline = run_tiled_baseline(INPUT_IMAGE_PATH, sess, TILE)
t_baseline = time.time() - t0

t0 = time.time()
_, out_overlap, n_tiles_overlap = run_tiled_overlap(INPUT_IMAGE_PATH, sess, TILE, OVERLAP)
t_overlap = time.time() - t0

print(f'Baseline (khong overlap): {n_tiles_baseline} tile, {t_baseline:.2f}s')
print(f'Overlap={OVERLAP}px       : {n_tiles_overlap} tile, {t_overlap:.2f}s ({n_tiles_overlap/n_tiles_baseline:.2f}x so tile)')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_in); axes[0].set_title('Input goc'); axes[0].axis('off')
axes[1].imshow(out_baseline); axes[1].set_title(f'Baseline: tile roi rac ({n_tiles_baseline} tile)'); axes[1].axis('off')
axes[2].imshow(out_overlap); axes[2].set_title(f'Overlap+blend ({n_tiles_overlap} tile)'); axes[2].axis('off')
plt.tight_layout()
plt.show()

diff_baseline_overlap = np.abs(out_baseline - out_overlap).mean()
print(f'Mean abs diff giua 2 cach ghep: {diff_baseline_overlap:.5f}')

os.makedirs('../Result/images', exist_ok=True)
Image.fromarray((out_baseline * 255).astype(np.uint8)).save('../Result/images/demo_baseline_no_overlap.png')
Image.fromarray((out_overlap * 255).astype(np.uint8)).save('../Result/images/demo_overlap_blend.png')
print('Da luu anh vao ../Result/images/')

## Buoc 4. Zoom vao vung ranh gioi tile de xem blend co giam duoc vet noi khong

Ranh gioi tile 256px trong baseline la boi so cua 256 (0, 256, 512, ...) — zoom dung vao
mot duong ranh gioi do (vi du quanh x=512) de xem baseline co "duong seam" nhin thay duoc
khong, va overlap+blend co lam mat duong do khong.

In [ ]:
CROP_Y, CROP_X, CROP_SIZE = 150, 450, 150  # <-- quanh ranh gioi tile x=512 (256*2)

crop_baseline = out_baseline[CROP_Y:CROP_Y+CROP_SIZE, CROP_X:CROP_X+CROP_SIZE, :]
crop_overlap = out_overlap[CROP_Y:CROP_Y+CROP_SIZE, CROP_X:CROP_X+CROP_SIZE, :]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(crop_baseline); axes[0].set_title('Baseline - ZOOM quanh ranh gioi tile'); axes[0].axis('off')
axes[1].imshow(crop_overlap); axes[1].set_title('Overlap+blend - ZOOM cung vung'); axes[1].axis('off')
plt.tight_layout()
plt.show()

## Buoc 5. Danh gia dinh luong: PSNR/SSIM tren tap test GoPro chinh thuc

So sanh 3 cach tren cung 1 tap anh: **input (chua xu ly)**, **baseline (tile roi rac)**,
**overlap+blend**. Doi chieu voi 2 moc da biet:
- Paper (`NAFNetLocal`, nguyen anh, co TLC): **33.71 dB**
- Baseline ONNX tile roi rac (da do truoc, full 1111 anh): **33.25 dB**

Nho da chuyen sang GPU o Buoc 1 (nhanh hon CPU ~15 lan), `N_SAMPLES = None` (chay full
1111 anh) de so sanh truc tiep 1-1 voi 2 con so tren — neu can chay nhanh hon de thu
truoc, doi lai thanh mot so nho (vi du 50).

In [ ]:
LMDB_DIR = '../Data/input.lmdb'
TARGET_LMDB_DIR = '../Data/target.lmdb'
N_SAMPLES = None  # <-- full 1111 anh (GPU du nhanh); doi thanh vd 50 neu muon thu nhanh

if not (os.path.exists(LMDB_DIR) and os.path.exists(TARGET_LMDB_DIR)):
    print(f'Chua tim thay lmdb test set trong {LMDB_DIR} / {TARGET_LMDB_DIR} — bo qua buoc nay.')
else:
    import lmdb
    import cv2
    from basicsr.metrics.psnr_ssim import calculate_psnr, calculate_ssim

    def decode_lmdb_image(env, key):
        lmdb_key = key.rsplit('.', 1)[0]  # lmdb luu key khong co duoi file (vd: 'GOPR0384_11_00-000001')
        with env.begin(write=False) as txn:
            buf = txn.get(lmdb_key.encode('ascii'))
        bgr = cv2.imdecode(np.frombuffer(buf, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        return rgb.astype(np.float32) / 255.0

    def tile_infer_baseline_array(img_np, sess, tile):
        h, w = img_np.shape[:2]
        pad_h, pad_w = (-h) % tile, (-w) % tile
        img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
        H_pad, W_pad = img_padded.shape[:2]
        out_padded = np.zeros_like(img_padded)
        for y in range(0, H_pad, tile):
            for x in range(0, W_pad, tile):
                tile_in = img_padded[y:y+tile, x:x+tile, :].transpose(2, 0, 1)[None].astype(np.float32)
                tile_out = sess.run(None, {'input': tile_in})[0]
                out_padded[y:y+tile, x:x+tile, :] = tile_out[0].transpose(1, 2, 0)
        return np.clip(out_padded[:h, :w, :], 0, 1)

    def tile_infer_overlap_array(img_np, sess, tile, overlap):
        h, w = img_np.shape[:2]
        pad_h, pad_w = max(0, tile - h), max(0, tile - w)
        img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
        H_pad, W_pad = img_padded.shape[:2]
        stride = tile - overlap
        ys = compute_positions(H_pad, tile, stride)
        xs = compute_positions(W_pad, tile, stride)
        weight2d = make_blend_weight(tile, overlap)
        out_acc = np.zeros((H_pad, W_pad, 3), dtype=np.float32)
        weight_acc = np.zeros((H_pad, W_pad, 1), dtype=np.float32)
        for y in ys:
            for x in xs:
                tile_in = img_padded[y:y+tile, x:x+tile, :].transpose(2, 0, 1)[None].astype(np.float32)
                tile_out = sess.run(None, {'input': tile_in})[0][0].transpose(1, 2, 0)
                out_acc[y:y+tile, x:x+tile, :] += tile_out * weight2d
                weight_acc[y:y+tile, x:x+tile, :] += weight2d
        out = out_acc / np.clip(weight_acc, 1e-6, None)
        return np.clip(out[:h, :w, :], 0, 1)

    meta_info_path = os.path.join(LMDB_DIR, 'meta_info.txt')
    with open(meta_info_path, 'r') as f:
        keys = [line.split('.png')[0] + '.png' for line in f.readlines()]
    sample_keys = keys if N_SAMPLES is None else keys[:N_SAMPLES]
    print(f'Danh gia tren {len(sample_keys)} / {len(keys)} anh cua tap test...')

    env_in = lmdb.open(LMDB_DIR, readonly=True, lock=False, readahead=False)
    env_gt = lmdb.open(TARGET_LMDB_DIR, readonly=True, lock=False, readahead=False)

    psnr_input_list, ssim_input_list = [], []
    psnr_baseline_list, ssim_baseline_list = [], []
    psnr_overlap_list, ssim_overlap_list = [], []

    t_baseline_total, t_overlap_total = 0.0, 0.0

    for idx, key in enumerate(sample_keys):
        img_in_np = decode_lmdb_image(env_in, key)
        img_gt_np = decode_lmdb_image(env_gt, key)

        t0 = time.time()
        out_baseline_np = tile_infer_baseline_array(img_in_np, sess, TILE)
        t_baseline_total += time.time() - t0

        t0 = time.time()
        out_overlap_np = tile_infer_overlap_array(img_in_np, sess, TILE, OVERLAP)
        t_overlap_total += time.time() - t0

        psnr_input_list.append(calculate_psnr(img_in_np, img_gt_np, crop_border=0))
        psnr_baseline_list.append(calculate_psnr(out_baseline_np, img_gt_np, crop_border=0))
        psnr_overlap_list.append(calculate_psnr(out_overlap_np, img_gt_np, crop_border=0))
        ssim_input_list.append(calculate_ssim(img_in_np, img_gt_np, crop_border=0))
        ssim_baseline_list.append(calculate_ssim(out_baseline_np, img_gt_np, crop_border=0))
        ssim_overlap_list.append(calculate_ssim(out_overlap_np, img_gt_np, crop_border=0))

        print(f'[{idx+1}/{len(sample_keys)}] {key}: PSNR input={psnr_input_list[-1]:.2f} '
              f'baseline={psnr_baseline_list[-1]:.2f} overlap={psnr_overlap_list[-1]:.2f}')

    env_in.close(); env_gt.close()

    print(f'\n=== KET QUA TRUNG BINH tren {len(sample_keys)} anh ===')
    print(f'Input (chua xu ly)      : PSNR = {np.mean(psnr_input_list):.2f} dB | SSIM = {np.mean(ssim_input_list):.4f}')
    print(f'Baseline (tile roi rac) : PSNR = {np.mean(psnr_baseline_list):.2f} dB | SSIM = {np.mean(ssim_baseline_list):.4f}')
    print(f'Overlap={OVERLAP}px + blend    : PSNR = {np.mean(psnr_overlap_list):.2f} dB | SSIM = {np.mean(ssim_overlap_list):.4f}')
    print(f'\nChenh lech overlap vs baseline: {np.mean(psnr_overlap_list) - np.mean(psnr_baseline_list):+.3f} dB')
    print(f'Thoi gian: baseline {t_baseline_total:.1f}s ({t_baseline_total/len(sample_keys):.2f}s/anh) | '
          f'overlap {t_overlap_total:.1f}s ({t_overlap_total/len(sample_keys):.2f}s/anh) '
          f'({t_overlap_total/t_baseline_total:.2f}x cham hon)')
    print(f'\n(Tham chieu: paper NAFNetLocal nguyen anh dat ~33.71 dB tren toan bo {len(keys)} anh test;')
    print(f' baseline tile roi rac tren toan bo {len(keys)} anh da do truoc do la 33.25 dB.)')

## Ket luan

- So sanh `Chenh lech overlap vs baseline` o Buoc 5 cho biet overlap+blend co giup
  tien gan hon ve 33.71 dB (paper) hay khong, va tien duoc bao nhieu.
- Doi chieu voi ty le thoi gian chay (`x cham hon`) de biet danh doi co dang gia khi
  trien khai vao app mobile that (nhieu tile hon -> nhieu lan forward hon -> cham hon
  va ton pin hon tren dien thoai).
- Neu muon do chinh xac tuyet doi (khong bi anh huong boi cache 50 anh dau), doi
  `N_SAMPLES = None` o Buoc 5 de chay full 1111 anh — se cho con so so sanh truc tiep
  1-1 voi 33.25 dB / 33.71 dB da co.
- Co the thu doi `OVERLAP` (32/64/96/128) o Buoc 3 va Buoc 5 de tim diem can bang tot
  nhat giua chat luong va tot do chay.